<a href="https://colab.research.google.com/github/Dinithiii04/AgriLak/blob/yield-component-cleanup/notebooks/YIELD_weather_dataset_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weather Dataset Preparation

In [20]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load Datasets in Dry Zone Districts

In [22]:
# Define the base directory path in Google Drive
base_path = "/content/drive/MyDrive/RAW Datasets/"

# Load each dataset with the correct path
battic_df = pd.read_csv(base_path + "BATTIC-RAW.csv")
jaffna_df = pd.read_csv(base_path + "JAFFNA-RAW.csv")
mannar_df = pd.read_csv(base_path + "MANNAR-RAW.csv")
monaragala_df = pd.read_csv(base_path + "MONARAGALA-RAW.csv")
mullativu_df = pd.read_csv(base_path + "MULLATIVU-RAW.csv")
pol_df = pd.read_csv(base_path + "POL-RAW.csv")
puttalam_df = pd.read_csv(base_path + "PUTTALAM-RAW.csv")
trinco_df = pd.read_csv(base_path + "TRINCO-RAW.csv")
vavuniya_df = pd.read_csv(base_path + "VAVUNIYA-RAW.csv")
hamb_df = pd.read_csv(base_path + "HAMB-RAW.csv")
amp_df = pd.read_csv(base_path + "AMP-RAW.csv")
anu_df = pd.read_csv(base_path + "ANU-RAW.csv")

# Print confirmation
print("All datasets loaded successfully!")

# Check the first few rows of any dataset
battic_df.head()


All datasets loaded successfully!


,PARAMETER,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANN
0,ALLSKY_SFC_SW_DWN,1984,12.71,11.65,19.75,18.61,21.00,20.14,17.48,19.53,19.81,18.66,16.61,15.23,17.60
1,ALLSKY_SFC_SW_DWN,1985,16.55,17.13,21.56,22.07,20.42,18.42,19.14,19.11,19.80,18.62,16.16,14.70,18.64
2,ALLSKY_SFC_SW_DWN,1986,14.12,18.44,18.68,22.73,22.01,19.18,18.94,20.23,19.81,18.73,16.67,14.35,18.66
3,ALLSKY_SFC_SW_DWN,1987,13.40,19.55,20.91,20.50,21.00,17.61,20.64,18.37,19.94,17.69,17.06,14.47,18.43
4,ALLSKY_SFC_SW_DWN,1988,14.78,18.51,19.64,19.07,18.91,19.97,17.07,17.66,18.00,20.78,15.41,14.94,17.89


## Merge all weather datasets

In [23]:
# Dictionary containing your already loaded datasets
datasets = {
    "BATTICALOA": battic_df,
    "JAFFNA": jaffna_df,
    "MANNAR": mannar_df,
    "MONARAGALA": monaragala_df,
    "MULLATIVU": mullativu_df,
    "POLONNARUWA": pol_df,
    "PUTTALAM": puttalam_df,
    "TRINCOMALEE": trinco_df,
    "VAVUNIYA": vavuniya_df,
    "HAMBANTOTA": hamb_df,
    "AMPARA": amp_df,
    "ANURADHAPURA": anu_df
}

# Define month mapping
month_mapping = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4, "MAY": 5, "JUN": 6,
    "JUL": 7, "AUG": 8, "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12
}


In [24]:
# Initialize an empty list to store structured data
structured_data = []

# Process each dataset
for district, df in datasets.items():
    # Drop the annual row
    df = df[df["YEAR"] != "ANN"]

    # Melt the dataframe to transform columns into rows
    df_melted = df.melt(id_vars=["PARAMETER", "YEAR"], var_name="MONTH", value_name="VALUE")

    # Remove annual values
    df_melted = df_melted[df_melted["MONTH"] != "ANN"]

    # Pivot table to structure data correctly
    df_pivot = df_melted.pivot(index=["YEAR", "MONTH"], columns="PARAMETER", values="VALUE")

    # Reset index properly
    df_pivot = df_pivot.reset_index()

    # Add district column
    df_pivot["District"] = district

    # Rename columns to match format
    df_pivot.rename(columns={
        "YEAR": "Year",
        "MONTH": "Month",
        "T2M_MAX": "Tmax",
        "T2M_MIN": "Tmin",
        "WS2M": "Wind",
        "ALLSKY_SFC_SW_DWN": "SRAD",
        "RH2M": "RH",
        "PRECTOTCORR_SUM": "Rain",
        "GWETROOT": "Soil"
    }, inplace=True)

    # Convert Month to numerical format using mapping
    df_pivot["Month"] = df_pivot["Month"].map(month_mapping)

    # Append to the list
    structured_data.append(df_pivot)

# Combine all districts into a single dataframe
final_df = pd.concat(structured_data, ignore_index=True)

# Sort by Year and Month again after conversion
final_df.sort_values(by=["Year", "Month"], inplace=True)

# Reset index after sorting
final_df.reset_index(drop=True, inplace=True)

# Reorder columns to match the specified order
final_df = final_df[["Year", "Month", "District", "Tmax", "SRAD", "RH", "Rain"]]


In [25]:
display(final_df.head())


PARAMETER,Year,Month,District,Tmax,SRAD,RH,Rain
0,1984,1,BATTICALOA,27.56,12.71,85.08,331.78
1,1984,1,JAFFNA,27.17,13.10,84.14,192.61
2,1984,1,MANNAR,27.97,14.18,87.13,254.00
3,1984,1,MONARAGALA,27.58,13.96,92.02,279.90
4,1984,1,MULLATIVU,26.89,13.10,84.98,266.93


## Exlore weather dataset

In [26]:
# Display basic information about the DataFrame
print("DataFrame Info:")
print(final_df.info())



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4608 entries, 0 to 4607
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Year      4608 non-null   int64  
 1   Month     4608 non-null   int64  
 2   District  4608 non-null   object 
 3   Tmax      4608 non-null   float64
 4   SRAD      4608 non-null   float64
 5   RH        4608 non-null   float64
 6   Rain      4608 non-null   float64
dtypes: float64(4), int64(2), object(1)
memory usage: 252.1+ KB
None


In [27]:
# Display summary statistics for numerical columns
print("\nSummary Statistics:")
print(final_df.describe())



Summary Statistics:
PARAMETER         Year        Month         Tmax         SRAD           RH  \
count      4608.000000  4608.000000  4608.000000  4608.000000  4608.000000   
mean       1999.500000     6.500000    32.197504    18.813166    77.525838   
std           9.234095     3.452427     2.678193     2.575499     5.929337   
min        1984.000000     1.000000    26.810000    10.780000    54.010000   
25%        1991.750000     3.750000    30.040000    17.107500    73.590000   
50%        1999.500000     6.500000    31.960000    19.280000    77.920000   
75%        2007.250000     9.250000    34.162500    20.730000    81.800000   
max        2015.000000    12.000000    41.500000    24.910000    93.450000   

PARAMETER         Rain  
count      4608.000000  
mean        107.554488  
std         108.313629  
min           0.000000  
25%          31.647500  
50%          71.305000  
75%         146.922500  
max         866.010000  


In [28]:
# Display column names
print("\nColumn Names:")
print(final_df.columns)



Column Names:
Index(['Year', 'Month', 'District', 'Tmax', 'SRAD', 'RH', 'Rain'], dtype='object', name='PARAMETER')


## Handle missin

In [29]:
# Display the number of missing values in each column
print("\nMissing Values:")
print(final_df.isnull().sum())


Missing Values:
PARAMETER
Year        0
Month       0
District    0
Tmax        0
SRAD        0
RH          0
Rain        0
dtype: int64


In [30]:
final_df.to_csv("/content/drive/MyDrive/NEW_monthly_weather_data.csv", index=False)
final_df.to_csv("/content/NEW_monthly_weather_data.csv", index=False)